In [0]:
import requests
from datetime import datetime

# ── CONFIG ──────────────────────────────────────────
APP_ID   = "768d0fe235e14d2c94070648b30acb2d"
BASE_URL = "https://openexchangerates.org/api/latest.json"

# ── FETCH FROM API ──────────────────────────────────
def fetch_exchange_rates():
    response = requests.get(f"{BASE_URL}?app_id={APP_ID}")
    if response.status_code == 200:
        print("API call successful")
        return response.json()
    else:
        raise Exception(f"API call failed: {response.status_code}")

# ── SAVE TO BRONZE ───────────────────────────────────
def save_to_bronze(data):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    rates = data["rates"]
    rows  = [{"currency": k, "rate": float(v), "base": str(data["base"]), "ingested_at": timestamp}
             for k, v in rates.items()]

    df = spark.createDataFrame(rows)

    spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

    df.writeTo("bronze.exchange_rates") \
        .using("delta") \
        .createOrReplace()

    print(f"Saved {df.count()} rows to bronze")

# ── RUN ─────────────────────────────────────────────
raw_data = fetch_exchange_rates()
save_to_bronze(raw_data)

# ── VERIFY ──────────────────────────────────────────
spark.sql("SELECT * FROM bronze.exchange_rates LIMIT 10").show()
print("Total rows:", spark.sql("SELECT COUNT(*) FROM bronze.exchange_rates").collect()[0][0])

In [0]:
# Check schema
spark.sql("DESCRIBE bronze.exchange_rates").show()

# Check for nulls
spark.sql("""
    SELECT 
        COUNT(*) as total_rows,
        COUNT(currency) as currency_count,
        COUNT(rate) as rate_count,
        MIN(rate) as min_rate,
        MAX(rate) as max_rate
    FROM bronze.exchange_rates
""").show()